In [1]:
import requests
import pandas as pd

API_KEY = "329e527fa42df701181726a287d8eee223378715"

url = (
    "https://api.census.gov/data/2023/acs/acs5"
    "?get=NAME,B19013_001E,B17001_001E,B17001_002E,"
    "B23025_003E,B23025_005E"
    "&for=county:*"
    "&in=state:*"
    f"&key={API_KEY}"
)

response = requests.get(url)

print(response.status_code)

data = response.json()

acs = pd.DataFrame(data[1:], columns=data[0])

print(acs.shape)
print(acs.head())

200
(3222, 8)
                      NAME B19013_001E B17001_001E B17001_002E B23025_003E  \
0  Autauga County, Alabama       69841       58731        6275       27070   
1  Baldwin County, Alabama       75019      236041       24819      113171   
2  Barbour County, Alabama       44290       21650        4746        9074   
3     Bibb County, Alabama       51215       20759        4258        9375   
4   Blount County, Alabama       61096       58665        8269       27180   

  B23025_005E state county  
0         688    01    001  
1        3615    01    003  
2         518    01    005  
3         936    01    007  
4        1586    01    009  


In [3]:
acs["fips"] = (
    acs["state"].astype(str).str.zfill(2)
    + acs["county"].astype(str).str.zfill(3)
)

print(acs[["NAME", "state", "county", "fips"]].head())
print(acs[["NAME", "state", "county", "fips"]].sample(5))

                      NAME state county   fips
0  Autauga County, Alabama    01    001  01001
1  Baldwin County, Alabama    01    003  01003
2  Barbour County, Alabama    01    005  01005
3     Bibb County, Alabama    01    007  01007
4   Blount County, Alabama    01    009  01009
                            NAME state county   fips
2501      Rhea County, Tennessee    47    143  47143
2193  Pittsburg County, Oklahoma    40    121  40121
1552   Moniteau County, Missouri    29    135  29135
1074     Martin County, Kentucky    21    159  21159
1027     Estill County, Kentucky    21    065  21065


In [4]:
acs = acs.rename(columns={
    "NAME": "county_name",
    "B19013_001E": "median_household_income",
    "B17001_001E": "poverty_universe",
    "B17001_002E": "below_poverty",
    "B23025_003E": "civilian_labor_force",
    "B23025_005E": "unemployed"
})

In [6]:
import pandas as pd

numeric_cols = [
    "median_household_income",
    "poverty_universe",
    "below_poverty",
    "civilian_labor_force",
    "unemployed"
]

for col in numeric_cols:
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs["poverty_rate"] = (
    acs["below_poverty"] /
    acs["poverty_universe"] * 100
)

acs["unemployment_rate"] = (
    acs["unemployed"] /
    acs["civilian_labor_force"] * 100
)

In [8]:
print(acs.dtypes)

county_name                 object
median_household_income      int64
poverty_universe             int64
below_poverty                int64
civilian_labor_force         int64
unemployed                   int64
state                       object
county                      object
fips                        object
poverty_rate               float64
unemployment_rate          float64
dtype: object


In [9]:
print(
    acs[
        [
            "county_name",
            "median_household_income",
            "poverty_rate",
            "unemployment_rate"
        ]
    ].head(10)
)

                county_name  median_household_income  poverty_rate  \
0   Autauga County, Alabama                    69841     10.684306   
1   Baldwin County, Alabama                    75019     10.514699   
2   Barbour County, Alabama                    44290     21.921478   
3      Bibb County, Alabama                    51215     20.511585   
4    Blount County, Alabama                    61096     14.095287   
5   Bullock County, Alabama                    36723     25.651127   
6    Butler County, Alabama                    44881     21.015866   
7   Calhoun County, Alabama                    55826     17.462188   
8  Chambers County, Alabama                    49295     15.480078   
9  Cherokee County, Alabama                    50769     14.523340   

   unemployment_rate  
0           2.541559  
1           3.194281  
2           5.708618  
3           9.984000  
4           5.835173  
5           5.873181  
6           6.790431  
7           6.438023  
8           4.961757  

In [10]:
acs_final = acs[
    [
        "fips",
        "county_name",
        "median_household_income",
        "poverty_rate",
        "unemployment_rate"
    ]
]

In [11]:
acs_final.to_csv(
    "ACS_2023_5Year_County_Socioeconomic.csv",
    index=False
)